In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

# ===============================
# CONSTANTS
# ===============================
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 8
EPOCHS_STAGE1 = 10
EPOCHS_STAGE2 = 10
DATASET_PATH = "/kaggle/input/datasets/amimulahasanrofik/alz-b-1100/alzheimer_new_11/alzheimer_new"

AUTOTUNE = tf.data.AUTOTUNE

2026-04-01 18:38:55.369862: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775068735.549490      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775068735.605409      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775068736.048976      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775068736.049018      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775068736.049021      55 computation_placer.cc:177] computation placer alr

In [3]:
# Get all file paths and labels
all_file_paths = []
all_labels = []

class_names = sorted(os.listdir(DATASET_PATH))
print("Class names:", class_names)

for label_idx, class_name in enumerate(class_names):
    class_dir = os.path.join(DATASET_PATH, class_name)
    for img_file in os.listdir(class_dir):
        all_file_paths.append(os.path.join(class_dir, img_file))
        all_labels.append(label_idx)

all_file_paths = np.array(all_file_paths)
all_labels = np.array(all_labels)

print("Total images:", len(all_file_paths))

Class names: ['MildDemented', 'Moderate Dementia', 'NonDemented', 'Very mild Dementia', 'glioma', 'meningioma', 'notumor', 'pituitary']
Total images: 8800


In [4]:
# Safe image decoding and resizing
def preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = img / 255.0
    return img, label

In [5]:
# Data augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.3),
    layers.RandomContrast(0.3),
    layers.RandomTranslation(0.1,0.1),
    layers.RandomBrightness(0.2)
])

# Mixup function
def mixup(images, labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    lambda_val = tf.random.uniform([],0,1)
    index = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lambda_val * images + (1-lambda_val) * tf.gather(images,index)
    labels_onehot = tf.one_hot(labels, NUM_CLASSES)
    mixed_labels = lambda_val * labels_onehot + (1-lambda_val) * tf.gather(labels_onehot,index)
    return mixed_images, mixed_labels

I0000 00:00:1775068759.709270      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [6]:
def transformer_block(x, head_size=64, num_heads=4, ff_dim=128):
    x1 = layers.LayerNormalization()(x)
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(x1, x1)
    x2 = layers.Add()([attn, x])
    x3 = layers.LayerNormalization()(x2)
    x3 = layers.Dense(ff_dim, activation="gelu")(x3)
    x3 = layers.Dense(head_size)(x3)
    return layers.Add()([x3, x2])

In [7]:
def build_model():
    base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224,224,3))
    base_model.trainable = False
    
    inputs = keras.Input(shape=(224,224,3))
    x = data_augmentation(inputs)
    x = base_model(x)
    
    shape = x.shape
    x = layers.Reshape((shape[1]*shape[2], shape[3]))(x)
    x = layers.Dense(64)(x)
    
    for _ in range(4):
        x = transformer_block(x)
    
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    
    model = keras.Model(inputs, outputs)
    return model

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
fold_no = 1

for train_index, val_index in skf.split(all_file_paths, all_labels):
    print(f"\n========== Fold {fold_no} ==========")
    
    train_paths, val_paths = all_file_paths[train_index], all_file_paths[val_index]
    train_labels, val_labels = all_labels[train_index], all_labels[val_index]
    
    train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
    train_ds = train_ds.map(preprocess).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    
    val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
    val_ds = val_ds.map(preprocess).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    
    model = build_model()
    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    callbacks = [
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
    ]
    
    # Stage 1 training
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_STAGE1, callbacks=callbacks)
    
    # Fine-tune last 50 layers
    base_model = model.layers[2]  # ResNet50 base
    for layer in base_model.layers[-50:]:
        layer.trainable = True
    model.compile(
        optimizer=keras.optimizers.Adam(1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_STAGE2, callbacks=callbacks)
    
    # ===============================
    # Metrics: Confusion Matrix & Classification Report
    # ===============================
    y_true = np.concatenate([y for x,y in val_ds], axis=0)
    y_pred = np.argmax(model.predict(val_ds), axis=1)
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix Fold {fold_no}")
    plt.show()
    
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    fold_no += 1


========== Fold 1 ==========
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/10


I0000 00:00:1775068789.661378     135 cuda_dnn.cc:529] Loaded cuDNN version 91002


248/248 ━━━━━━━━━━━━━━━━━━━━ 54s 123ms/step - accuracy: 0.3681 - loss: 1.7475 - val_accuracy: 0.1261 - val_loss: 2.9531 - learning_rate: 1.0000e-04
Epoch 2/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 28s 110ms/step - accuracy: 0.3255 - loss: 1.9306 - val_accuracy: 0.2045 - val_loss: 2.3627 - learning_rate: 1.0000e-04
Epoch 3/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 28s 109ms/step - accuracy: 0.2951 - loss: 2.0407 - val_accuracy: 0.1886 - val_loss: 2.9611 - learning_rate: 1.0000e-04
Epoch 4/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 28s 110ms/step - accuracy: 0.2876 - loss: 1.9984 - val_accuracy: 0.1614 - val_loss: 2.3249 - learning_rate: 1.0000e-04
Epoch 5/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 28s 111ms/step - accuracy: 0.3524 - loss: 1.8264 - val_accuracy: 0.2761 - val_loss: 1.8758 - learning_rate: 1.0000e-04
Epoch 6/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 28s 110ms/step - accuracy: 0.3585 - loss: 1.7978 - val_accuracy: 0.2091 - val_loss: 2.1840 - learning_rate: 1.0000e-04
Epoch 7/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 28s 110ms/step - 

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name='conv5_block3_out', pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        loss = predictions[:, pred_index]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap,0)/tf.math.reduce_max(heatmap)
    return heatmap

# Example usage
import cv2
sample_img, _ = preprocess(all_file_paths[0], all_labels[0])
sample_img_array = np.expand_dims(sample_img, axis=0)
heatmap = make_gradcam_heatmap(sample_img_array, model)

# Overlay heatmap
img = sample_img.numpy()
heatmap = cv2.resize(heatmap.numpy(), (img.shape[1], img.shape[0]))
heatmap = np.uint8(255 * heatmap)
heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
superimposed_img = heatmap*0.4 + img*255
plt.imshow(np.uint8(superimposed_img))
plt.axis("off")
plt.show()